In [ ]:
repository_filter: list[str] = []
data_file_methods: str = "../samples/method_quality_metrics.csv"
data_file_packages: str = "../samples/package_quality_metrics.csv"
data_file_smells: str = "../samples/code_smells.csv"
data_file_gaps: str = "../samples/test_gaps.csv"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import warnings

warnings.simplefilter("ignore")

# Primary data table (injected via NB_DATA_TABLE env var)
class_df = dt.read_csv("../samples/class_quality_metrics.csv")
class_df = qu.filter_repos(class_df, repository_filter)
class_df = qu.add_repo_short(class_df)

# Additional data tables (injected via papermill parameters)
methods_df = pd.read_csv(data_file_methods, on_bad_lines="skip")
methods_df = qu.filter_repos(methods_df, repository_filter)
methods_df = qu.add_repo_short(methods_df)

pkg_df = pd.read_csv(data_file_packages, on_bad_lines="skip")
pkg_df = qu.filter_repos(pkg_df, repository_filter)

smells_df = pd.read_csv(data_file_smells, on_bad_lines="skip")
smells_df = qu.filter_repos(smells_df, repository_filter)
smells_df = qu.add_repo_short(smells_df)

gaps_df = pd.read_csv(data_file_gaps, on_bad_lines="skip")
gaps_df = qu.filter_repos(gaps_df, repository_filter)
gaps_df = qu.add_repo_short(gaps_df)

In [ ]:
has_data = len(class_df) > 0

if not has_data:
    fig = qu.empty_figure("No data available for the executive dashboard.")
else:
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            "Code Health by Repository<br><sup>Composite of method complexity, code volume, and information density per class (Maintainability Index)</sup>",
            "Top Refactoring Targets",
        ],
        specs=[
            [{"type": "bar"}],
            [{"type": "table"}],
        ],
        vertical_spacing=0.08,
        row_heights=[0.55, 0.45],
    )

    # --- Panel 1: Code Health by Repository (best 5, gap, worst 5) ---
    repo_health = (
        class_df.groupby("repoShort")["maintainabilityIndex"]
        .mean()
        .sort_values(ascending=False)
    )
    total_repos = len(repo_health)
    n_show = min(5, total_repos)

    if total_repos <= 10:
        display_health = repo_health.sort_values(ascending=True)
        display_labels = list(display_health.index)
    else:
        best = repo_health.head(n_show).sort_values(ascending=True)
        worst = repo_health.tail(n_show).sort_values(ascending=True)
        elided = total_repos - 2 * n_show
        display_vals = pd.concat([worst, pd.Series({f"··· {elided} more repos ···": repo_health.median()}), best])
        display_health = display_vals
        display_labels = list(display_health.index)

    colors = []
    for label, v in display_health.items():
        if "··· " in str(label):
            colors.append("#E0E0E0")
        elif v < 50:
            colors.append("#EF5350")
        elif v < 65:
            colors.append("#FF8A65")
        else:
            colors.append("#4CAF50")

    texts = []
    for label, v in display_health.items():
        if "··· " in str(label):
            texts.append("")
        else:
            texts.append(f"{v:.0f}")

    fig.add_trace(
        go.Bar(
            y=display_labels,
            x=display_health.values.round(1),
            orientation="h",
            marker_color=colors,
            text=texts,
            textposition="outside",
            showlegend=False,
        ),
        row=1, col=1,
    )
    fig.update_xaxes(
        title_text="Health score (higher is better)",
        title_font=dict(size=12),
        range=[0, 110],
        row=1, col=1,
    )

    # --- Panel 2: Top Refactoring Targets ---
    ranked = class_df.copy()
    ranked["priority"] = (100 - ranked["maintainabilityIndex"]) + ranked["wmc"]
    top_targets = ranked.nlargest(15, "priority")

    def health_label(mi):
        if mi < 50:
            return "Poor"
        elif mi < 65:
            return "Fair"
        return "Good"

    fig.add_trace(
        go.Table(
            columnwidth=[120, 160, 70, 70, 100],
            header=dict(
                values=["Repository", "Class", "Health", "Methods", "Should Split Into"],
                fill_color="#7986CB",
                font=dict(color="white", size=12),
                align="left",
                height=30,
            ),
            cells=dict(
                values=[
                    top_targets["repoShort"] if "repoShort" in top_targets.columns else top_targets["repositoryPath"].apply(qu.short_repo),
                    top_targets["className"].apply(qu.short_class),
                    top_targets["maintainabilityIndex"].apply(lambda x: f"{health_label(x)} ({x:.0f})"),
                    top_targets["methodCount"],
                    top_targets.apply(lambda r: f"Split into {r['lcom4']} classes" if r["lcom4"] > 1 else ("Reduce complexity" if r["wmc"] > 30 else "Simplify"), axis=1),
                ],
                align="left",
                font=dict(size=12),
                height=28,
            ),
        ),
        row=2, col=1,
    )

    fig.update_layout(
        title=dict(text="Code Quality Executive Dashboard", font=dict(size=22)),
        height=950,
        width=1000,
        margin=dict(l=180, r=50, t=80, b=30),
        plot_bgcolor="white",
    )
fig.show()